# STARCOP training on Google Colab (T4, Environment A)

Runs `src/training/train.py` -- the same entrypoint `train_mac.sh`/`train_desktop.sh` use -- on a free-tier Colab GPU. See `mlops-methane-detection-plan.md` TASK-3.3c for the full design and `internal-docs/setup/environment-notes.md`'s Colab sections for how every fix below was found.

**Before running anything else: Runtime → Change runtime type → Hardware accelerator → GPU (T4).** This isn't just about getting a GPU -- Colab's CPU-only runtime defaults to a newer Python that has no `torch==1.13.1` wheel at all. Picking the GPU runtime also switches to a Python version that does. Leaving this on CPU (or picking the wrong runtime) breaks the install cells below in a confusing way, not a clean error.

**Run cells top-to-bottom, in order, in a single kernel session.** Later cells depend on state earlier cells set (the cloned repo becoming the working directory, installed packages, `sys.path`) -- running a cell standalone, or after a kernel restart without re-running everything above it, produces confusing errors like `ModuleNotFoundError: No module named 'colab_bootstrap'` that look like a real bug but just mean an earlier cell's effect on this kernel is gone. If that happens, re-run from section 1.

## 1. Clone the repo

In [11]:
!git clone --recurse-submodules https://github.com/douglas-martins/methane-detection.git
%cd methane-detection

Cloning into 'methane-detection'...
remote: Enumerating objects: 1894, done.
remote: Counting objects: 100% (302/302), done.
remote: Compressing objects: 100% (197/197), done.
remote: Total 1894 (delta 148), reused 242 (delta 105), pack-reused 1592 (from 1)
Receiving objects: 100% (1894/1894), 35.51 MiB | 18.81 MiB/s, done.
Resolving deltas: 100% (1018/1018), done.
Submodule 'vendor/starcop' (https://github.com/spaceml-org/STARCOP) registered for path 'vendor/starcop'
Cloning into '/content/methane-detection/methane-detection/vendor/starcop'...
remote: Enumerating objects: 176, done.        
remote: Counting objects: 100% (45/45), done.        
remote: Compressing objects: 100% (16/16), done.        
remote: Total 176 (delta 37), reused 33 (delta 29), pack-reused 131 (from 1)        
Receiving objects: 100% (176/176), 10.61 MiB | 13.85 MiB/s, done.
Resolving deltas: 100% (60/60), done.
Submodule path 'vendor/starcop': checked out 'c4789268a3fa0395f92357429052f6f5fc748acb'
/content/meth

## 2. Dataset name

Defaults to `starcop_mini` (342 MB). `starcop_raw` (59 GB / 75k files) works but is slow on Colab's ephemeral disk, and each fresh session re-pulling it repeats the Google Drive API call volume flagged as a rate-limit risk (D-01) -- expect it to take a while, and avoid re-running this notebook top-to-bottom repeatedly against `starcop_raw` in a short window.

In [12]:
DATASET_NAME = "starcop_mini"

## 3. Install Environment A's pinned stack

Exact order confirmed by TASK-3.3c's step 1 spike -- reordering or combining these differently reintroduces bugs already found once (see `internal-docs/setup/environment-notes.md`):

1. `pip<24.1` -- Colab's default pip refuses `pytorch-lightning==1.6.4`'s malformed wheel metadata.
2. `numpy<2` -- must land before `torch` touches a real tensor, or torch's numpy interop breaks (`_ARRAY_API not found`).
3. `torch==1.13.1` -- has a `cp311` wheel on the GPU runtime's Python 3.11.13 (no separate Python provisioning needed).
4. The rest of `vendor/starcop/requirements.txt`'s pins, plus `requirements/env-a-mlflow.txt`'s `mlflow<3.7`/`boto3`/`protobuf<4` (mlflow's `torch.export` import bug and wandb's old-style `_pb2.py` files, same reasons documented there).

In [13]:
!python -m pip install -q "pip<24.1"
!pip --version

DEPRECATION: pytorch-lightning 1.6.4 has a non-standard dependency specifier torch>=1.8.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
pip 24.0 from /usr/local/lib/python3.11/dist-packages/pip (python 3.11)


In [14]:
!pip install -q "numpy<2"
!pip install -q torch==1.13.1

DEPRECATION: pytorch-lightning 1.6.4 has a non-standard dependency specifier torch>=1.8.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
DEPRECATION: pytorch-lightning 1.6.4 has a non-standard dependency specifier torch>=1.8.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytorch-lightning 1.6.4 requires protobuf<=3.20.1, but you 

In [15]:
!pip install -q \
  "pytorch-lightning==1.6.4" \
  "torchmetrics==0.10.0" \
  "kornia==0.6.7" \
  "wandb==0.13.3" \
  segmentation_models_pytorch \
  "setuptools<81" \
  "protobuf<4" \
  "mlflow<3.7" \
  boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 16.2 MB/s eta 0:00:00a 0:00:01
DEPRECATION: pytorch-lightning 1.6.4 has a non-standard dependency specifier torch>=1.8.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
asyncssh 2.24.0 requires cryptography>=48.0.1, but you have cryptography 46.0.7 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 3.20.1 which is incompatible.
google-api-core 2.25.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.

## 4. Install DVC

Same version this project pins (`uv.lock`), plus the two fixes TASK-3.3c's step 1b spike found: `protobuf<4` must be pinned in the *same* command as the `dvc[gdrive]` install (a separate later command re-resolves it upward and breaks wandb/pytorch-lightning again), and `pyOpenSSL` has to be removed entirely rather than upgraded or downgraded -- `pydrive2`'s legacy `oauth2client` auth path has no single `pyOpenSSL`/`cryptography` version pairing that satisfies both `cryptography`'s newer API and `oauth2client`'s `crypto.sign()` call. Removing `pyOpenSSL` forces `oauth2client` onto its pure-Python RSA signer instead, which has no C-extension ABI to break.

In [16]:
!pip install -q "dvc[gdrive]==3.67.1" "protobuf<4"
!pip uninstall -y pyOpenSSL
!pip install -q rsa pyasn1-modules

DEPRECATION: pytorch-lightning 1.6.4 has a non-standard dependency specifier torch>=1.8.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytorch-lightning 1.6.4 requires protobuf<=3.20.1, but you have protobuf 3.20.3 which is incompatible.
mlflow 3.6.0 requires cryptography<47,>=43.0.0, but you have cryptography 50.0.0 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompati

## 5. Credentials

**In the Colab web UI** (colab.research.google.com): one-time setup, per Google account -- click the 🔑 key icon in the left sidebar → **Secrets**, and add each of the following, with **Notebook access** enabled. None of these are ever written into this `.ipynb` file.

**Running via VS Code's Colab connection instead**: `google.colab.userdata.get()` can't reach the Secrets panel from there (no browser frontend for its RPC) -- and a `Colab: Open Terminal` shell is a *separate process* from this kernel, so `export`ing a variable there never reaches `os.environ` here either. Use a file instead, loaded directly into this kernel's own process:

- Create `.env.mlflow` in the cloned repo root (`/content/methane-detection/.env.mlflow`, git-ignored -- the same file/convention `train_mac.sh`/`train_desktop.sh` already use) with the six `MLFLOW_*`/`AWS_*` lines below, `KEY=value` per line, plus `WANDB_API_KEY=...` if wanted. Easiest via VS Code's file explorer, since it's browsing this same Colab VM's filesystem.
- For `DVC_GDRIVE_SERVICE_ACCOUNT_JSON`, don't put the JSON blob in `.env.mlflow` -- place the key file itself directly at `/content/gdrive-service-account.json` (drag it in via VS Code's file explorer, or copy the same key file already used by the `mac-mps` worker). The next cell checks for that file before asking for a secret/env var at all.

| Secret / env var name | Value |
|---|---|
| `MLFLOW_TRACKING_URI` | `https://methane-detection-mlflow.ghostface.tech` |
| `MLFLOW_TRACKING_USERNAME` | MLflow basic-auth username |
| `MLFLOW_TRACKING_PASSWORD` | MLflow basic-auth password |
| `MLFLOW_S3_ENDPOINT_URL` | B2 artifact store endpoint |
| `AWS_ACCESS_KEY_ID` | Dedicated client-side B2 Application Key ID (see `internal-docs/setup/environment-notes.md`) |
| `AWS_SECRET_ACCESS_KEY` | That key's secret |
| `DVC_GDRIVE_SERVICE_ACCOUNT_JSON` | Colab web UI only -- full contents of the existing D-01 service-account JSON key (**reuse the key already created for the `mac-mps` worker**, don't mint a new one). Via VS Code, place the key file directly instead (see above). |
| `WANDB_API_KEY` (optional) | Only if you want real W&B logging from Colab; otherwise training runs with `WANDB_MODE=disabled` (D-09) and MLflow logging is unaffected |

In [17]:
import os
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "training" / "colab_bootstrap.py").exists():
    raise RuntimeError(
        f"src/training/colab_bootstrap.py not found under {REPO_ROOT}. Run section 1's "
        "clone cell first, or check this kernel's working directory with `!pwd` -- it "
        "must be the cloned methane-detection repo root. This usually means an earlier "
        "cell was never run in this kernel session (or the kernel restarted)."
    )
sys.path.insert(0, str(REPO_ROOT / "src" / "training"))

try:
    from google.colab import userdata

    userdata_get = userdata.get
except ImportError:
    # No google.colab module reachable -- e.g. a VS Code kernel attached to this
    # Colab runtime with no browser frontend to service the Secrets RPC. Load a
    # local .env.mlflow instead, directly into *this* kernel's os.environ -- a
    # `Colab: Open Terminal` shell is a separate process, so `export`ing a
    # variable there would never reach this process.
    userdata_get = None
    !pip install -q python-dotenv
    from dotenv import load_dotenv

    load_dotenv(REPO_ROOT / ".env.mlflow")

import colab_bootstrap  # noqa: E402
import launch_profiles  # noqa: E402

DVC_JSON_SECRET = "DVC_GDRIVE_SERVICE_ACCOUNT_JSON"

# Fail loudly with the missing credential's name, same as train_mac.sh/train_desktop.sh's
# pre-flight check -- don't let a missing credential surface later as an opaque subprocess
# failure. DVC_JSON_SECRET is skipped here: cell below checks for an already-placed key
# file before falling back to a secret/env var for it.
for secret_name in colab_bootstrap.required_colab_secrets():
    if secret_name == DVC_JSON_SECRET:
        continue
    value = colab_bootstrap.read_secret(secret_name, userdata_get, os.environ)
    if not value:
        raise RuntimeError(
            f"Required credential {secret_name!r} is not set. In the Colab web UI, add "
            "it via the \U0001f511 key icon in the left sidebar (Notebook access enabled). "
            "Via VS Code, add it to .env.mlflow in the repo root instead (see the markdown "
            "cell above)."
        )
    os.environ[secret_name] = value

# Public, non-secret -- hardcoded rather than trusted from a Secret/env var, same
# reasoning as train_mac.sh/train_desktop.sh's MLFLOW_TRACKING_URI.
os.environ["MLFLOW_TRACKING_URI"] = "https://methane-detection-mlflow.ghostface.tech"

print("Credentials loaded (values not printed).")

RuntimeError: src/training/colab_bootstrap.py not found under /content/methane-detection/methane-detection. Run section 1's clone cell first, or check this kernel's working directory with `!pwd` -- it must be the cloned methane-detection repo root. This usually means an earlier cell was never run in this kernel session (or the kernel restarted).

In [ ]:
key_path = Path("/content/gdrive-service-account.json")
if key_path.exists():
    print(f"Using existing service-account key already at {key_path}.")
else:
    key_json = colab_bootstrap.read_secret(DVC_JSON_SECRET, userdata_get, os.environ)
    if not key_json:
        raise RuntimeError(
            f"{DVC_JSON_SECRET} is not set, and no key file already exists at {key_path}. "
            "In the Colab web UI, add it as a Secret. Via VS Code, place the key file "
            f"directly at {key_path} instead (see the markdown cell above) -- simpler "
            "than putting the JSON blob in .env.mlflow as a single line."
        )
    key_path.write_text(key_json)
os.chmod(key_path, 0o600)

for command in colab_bootstrap.dvc_service_account_setup_commands(str(key_path)):
    subprocess.run(command, check=True)

print("DVC configured for the service account (no OAuth prompt expected).")

In [ ]:
# D-09 parity with train_mac.sh/train_desktop.sh: default to WANDB_MODE=disabled
# when no WANDB_API_KEY secret was set, so training doesn't hang on an interactive
# wandb.login() prompt with no TTY. An explicit WANDB_MODE or a real key both win.
wandb_mode = colab_bootstrap.resolve_wandb_mode(
    wandb_api_key=os.environ.get("WANDB_API_KEY"),
    requested_mode=os.environ.get("WANDB_MODE"),
)
if wandb_mode is not None:
    os.environ["WANDB_MODE"] = wandb_mode
print("WANDB_MODE:", os.environ.get("WANDB_MODE", "<unset, real key present>"))

## 6. Pull the dataset

`data/processed/<dataset>` is **not** a valid pull target -- it's only the parent of `dvc.yaml`'s five separate stage outputs, confirmed by TASK-3.3c's step 1b spike (`NoOutputOrStageError`). Pull the raw dataset directly, then the real processed-pipeline outputs by their `dvc.yaml` foreach-stage names.

In [ ]:
!dvc pull data/{DATASET_NAME} -v

In [ ]:
!dvc pull \
  "normalize@{DATASET_NAME}" \
  "split@{DATASET_NAME}" \
  "patch_extract@{DATASET_NAME}" \
  "stats@{DATASET_NAME}" \
  "coordinates@{DATASET_NAME}" \
  -v

## 7. Train

Same `launch_profiles.build_launch_args` function `train_mac.sh`/`train_desktop.sh` use, so the `colab` machine profile (`training.accelerator=gpu training.devices=1`) stays a single source of truth across all three launch paths.

In [ ]:
launch_args = launch_profiles.build_launch_args("colab", DATASET_NAME)
print("train.py", *launch_args)

subprocess.run(
    [sys.executable, "src/training/train.py", *launch_args],
    check=True,
)